# Tutorial 10: Detecting Filamentous Fungi

Filamentous fungi like *Neurospora* grow as branching networks of hyphae,
not compact round colonies. Standard threshold detectors struggle with
this morphology because hyphae are thin and have varying intensity.
PhenoTypic includes a specialized pipeline for exactly this scenario.

**What you will learn:**

1. Load a filamentous fungi plate image
2. See why standard detectors miss thin hyphae
3. Use `FilamentousFungiPipeline` for end-to-end processing
4. Visualize fungal detection results

## Imports

In [ ]:
import phenotypic as pht
from phenotypic.data import load_fungi_plate

## Load the Fungi Plate

PhenoTypic ships a *Neurospora* filamentous fungi plate image. Let's
take a look at the spreading hyphal morphology.

In [ ]:
plate = load_fungi_plate()
plate.dash()

Notice the spreading mycelium — thin, branching filaments radiating from
each inoculation point. This is very different from the compact round
yeast colonies we have worked with so far.

## Why Standard Detectors Struggle

Let's see what happens when we apply `OtsuDetector` to this plate.

In [ ]:
from phenotypic.detect import OtsuDetector

test = plate.copy()
test = OtsuDetector()(test)
test.dash(overlay=True)

The Otsu threshold captures the dense central regions but misses the
thinner outer hyphae. This is expected — a single global threshold
cannot separate thin filaments from background when their intensities
overlap.

## Use FilamentousFungiPipeline

`FilamentousFungiPipeline` is specifically designed for branching fungal
morphology. It chains:

1. **BM3D denoising** — removes noise while preserving thin filaments
2. **Homomorphic filtering** — corrects uneven illumination
3. **FilamentousFungiDetector** — two-stage detection with Dijkstra
   reconnection to capture fragmented hyphal branches

In [ ]:
from phenotypic.prefab import FilamentousFungiPipeline

plate = load_fungi_plate()
fungi_pipeline = FilamentousFungiPipeline()
plate = fungi_pipeline.apply(plate)

## View the Results

In [ ]:
plate.dash(overlay=True)

Much better! The specialized pipeline captures the full extent of the
mycelium, including thin outer branches that the Otsu detector missed.

In [ ]:
plate.dash(overlay=True, show_gridlines=True)

In [ ]:
print(f"Detected fungal colonies: {plate.objmap.num_objects}")

## A Note on FrangiVesselness

If you want to build a custom pipeline for filamentous organisms, the
`FrangiVesselness` enhancer is a useful building block. It enhances
tubular structures (hyphae, branches) in `detect_mat` using Hessian-based
vesselness filtering.

```python
from phenotypic.enhance import FrangiVesselness

frangi = FrangiVesselness(sigmas=(0.5, 1.0, 1.5), black_ridges=False)
```

The `FilamentousFungiPipeline` uses a similar approach internally, combined
with phase congruency and minimum-cost path reconnection.

## Summary

You have detected filamentous fungi using PhenoTypic's specialized pipeline:

- **Standard detectors** (like `OtsuDetector`) miss thin hyphae because a single
  threshold cannot capture varying filament intensities.
- **`FilamentousFungiPipeline`** handles this with BM3D denoising, homomorphic
  filtering, and a two-stage detector with Dijkstra reconnection.
- **`FrangiVesselness`** is available for custom pipeline building.

---

**Congratulations — you have completed the tutorial series!** You now know
how to:

1. Load and inspect plate images
2. Detect colonies with thresholding and specialized detectors
3. Enhance images before detection
4. Build reusable pipelines
5. Work with grid plates
6. Batch-process many plates from the CLI
7. Measure and export colony features
8. Use prefab pipelines for common scenarios
9. Assess image quality
10. Handle filamentous fungi

For task-specific recipes, head to the [How-To Guides](../../how_to/index.rst).
For deeper understanding, see the [Explanation](../../explanation/index.rst) pages.